# Enterprise AI Knowledge Assistant: High-Performance RAG Pipeline & Benchmarking

Welcome to this enterprise-grade, experiment-oriented Google Colab notebook. In this project, we implement and benchmark a complete **Retrieval-Augmented Generation (RAG)** system designed to operate under real-world constraints, such as limited GPU memory (VRAM) and multi-lingual user requirements.

This notebook is structured as an ML systems engineering project and covers the full lifecycle of a RAG pipeline: document ingestion, pre-processing, multi-model embedding generation, vector search indexing, quantized LLM inference, prompt optimization, benchmarking, and failure analysis.

---

### Conceptual Architecture

Below is the conceptual flow of our Enterprise RAG pipeline:

```
                  +--------------------------------+
                  |  Enterprise Document Corpus    |
                  | (HR, IT, Compliance, Multi-L)  |
                  +---------------+----------------+
                                  |
                                  v
                  +---------------+----------------+
                  |  Document Preprocessing &      |
                  |   Chunking (Sliding Window)    |
                  +---------------+----------------+
                                  |
                                  v
                  +---------------+----------------+
                  |   Embedding Generation Model   |
                  | (MiniLM vs BGE-Small vs E5)    |
                  +---------------+----------------+
                                  |
                                  v
                  +---------------+----------------+
                  |     FAISS Vector Database      |
                  |   (Flat Index vs IVF Index)    |
                  +---------------+----------------+
                                  ^
                                  | Query Embeddings
  +--------------+                |
  | User Query   +----------------+
  +--------------+
        |
        v
  +---------------+----------------+
  | Grounded Prompt Construction   | <-- Top-K Chunks Retrieved
  +---------------+----------------+
                                  |
                                  v
                  +---------------+----------------+
                  |    Quantized LLM Inference     |
                  |  (FP16 vs INT8 vs INT4 TinyL)  |
                  +---------------+----------------+
                                  |
                                  v
                  +---------------+----------------+
                  |     Verified Output Response   |
                  +--------------------------------+
```

---

### Core Engineering Concepts Covered

#### 1. What is Retrieval-Augmented Generation (RAG)?
Retrieval-Augmented Generation (RAG) is a pattern that combines the parametric knowledge of a pre-trained Large Language Model (LLM) with external, non-parametric knowledge from an arbitrary document database. Rather than attempting to fine-tune an LLM on dynamic, proprietary company data, we query a vector database at inference time, retrieve highly relevant document chunks, and inject them directly into the context window of the LLM.

#### 2. The Enterprise Hallucination Problem
LLMs generate text based on statistical probabilities of next-tokens. While they excel at syntax and reasoning, they lack a concept of "truth" and are highly prone to **hallucinations**—confidently stating incorrect or fabricated facts. In enterprise settings (e.g. HR benefits, legal compliance, or healthcare guidelines), hallucinations can cause severe operational, legal, and financial damage. RAG mitigates this by restricting the model's source of truth to provided documents.

#### 3. Why Retrieval Optimization Matters
A RAG system is only as good as its retrieval engine. If the vector search fails to retrieve the correct chunk, or retrieves noisy, unrelated chunks, the LLM will generate either incorrect or low-quality answers. We will explore key retrieval hyper-parameters:
- **Chunk Size & Overlap:** Affecting semantic density and context retention.
- **Embedding Models:** Balancing representation capability, dimensions, and cross-lingual alignment.
- **Indexing Algorithms:** Comparing exact, brute-force search (`IndexFlat`) with fast, approximate nearest neighbor search (`IndexIVF`).

#### 4. The Critical Role of LLM Quantization
State-of-the-art LLMs contain billions of parameters. Loading a standard 7-billion parameter model in 16-bit floating-point precision (FP16) requires roughly **14–15 GB of VRAM** just for model weights, which exceeds the memory capacity of the standard Google Colab T4 GPU (16 GB) once context cache and batch dimensions are factored in.
**Quantization** (FP16 -> INT8 -> INT4) compresses these weights by mapping continuous floating-point ranges to discrete integer values. This reduces the memory footprint of a 7B model from ~15GB to ~4.5GB (INT4), enabling high-speed local inference on commodity hardware without severe degradation in reasoning quality.


## 2. Environment Setup

We begin by installing all required dependencies. This suite includes libraries for document processing (`pymupdf`), embeddings (`sentence-transformers`), indexing (`faiss-cpu`), quantized model loading (`bitsandbytes` + `accelerate` + `transformers`), and evaluation (`pandas`, `numpy`, `matplotlib`, `scikit-learn`).

*Note: For maximum performance, if a GPU is available, BitsAndBytes will automatically bind to CUDA. We will build robust VRAM monitoring helpers.*


In [ ]:
# Install required packages quietly
!pip install -q transformers sentence-transformers faiss-cpu bitsandbytes accelerate datasets pymupdf pandas numpy matplotlib scikit-learn langchain-community torch

import torch
import sys
import os
import gc
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("="*60)
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: CUDA is not available. BitsAndBytes and GPU acceleration will be disabled.")
print("="*60)

# VRAM Monitoring Utilities
def get_vram_usage():
    """
    Tracks and prints current GPU memory usage metrics using PyTorch.
    Returns:
        allocated (float): Memory currently allocated by tensors in MB.
        reserved (float): Total memory managed by the caching allocator in MB.
    """
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 2)
        reserved = torch.cuda.memory_reserved() / (1024 ** 2)
        print(f"[VRAM Monitor] Allocated: {allocated:.2f} MB | Reserved: {reserved:.2f} MB")
        return allocated, reserved
    else:
        print("[VRAM Monitor] CUDA is not active. VRAM tracking unavailable.")
        return 0.0, 0.0

def flush_memory():
    """
    Performs garbage collection and empties the PyTorch CUDA memory cache.
    Useful for freeing memory between model loads.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("[Memory Flush] GPU memory cache cleared.")
    else:
        print("[Memory Flush] CPU garbage collected.")

# Initial check
get_vram_usage()


## 3. Dataset Creation

To evaluate our RAG assistant, we will construct a high-quality, synthetic enterprise corpus. This corpus mimics a multinational corporation's internal knowledge base and covers:
1. **HR Policies** (leave systems, flexible hours, stipends)
2. **IT Troubleshooting Guidelines** (VPN connection errors, password policies)
3. **Onboarding Manuals** (new hire task list, development environment SOP)
4. **Compliance & Legal Documents** (anti-bribery, data privacy / GDPR)
5. **Healthcare Benefits FAQ** (copays, mental health programs)
6. **Finance Guidelines** (travel expense reporting, flight booking)
7. **Multilingual English and Hindi Documents** (same policy written in both languages)


In [ ]:
documents = [
    {
        "id": "doc_hr_01",
        "category": "HR Policies",
        "title": "Annual Leave and Time-Off Policy",
        "content": "Employees are entitled to 25 days of paid annual leave per calendar year. Vacation requests must be submitted through the HR Portal at least 14 days in advance and approved by the direct manager. Unused leave up to a maximum of 5 days can be carried forward to the next calendar year, but must be utilized within the first quarter (by March 31st). Failure to use the carried-over leave by this deadline results in automatic forfeiture. Compassionate leave is granted up to 5 days for immediate family members.",
        "language": "English"
    },
    {
        "id": "doc_hr_02",
        "category": "HR Policies",
        "title": "Remote Work and Flexible Hours Guideline",
        "content": "Our remote work framework supports a hybrid model requiring at least 2 days of in-office presence per week. Core collaboration hours are set between 10:00 AM and 4:00 PM EST, during which all employees must be reachable via corporate Slack and email. A home office stipend of $500 is provided as a one-time reimbursement for ergonomic chairs, monitors, and high-speed internet setup. Expense reports for ergonomic equipment must be submitted within 60 days of purchase with valid receipts.",
        "language": "English"
    },
    {
        "id": "doc_it_01",
        "category": "IT Support",
        "title": "Corporate VPN Connection Troubleshooting Guide",
        "content": "To connect to the corporate network remotely, use the Cisco AnyConnect client. Enter the gateway address 'vpn.enterprise.global.com'. Authentication requires your standard active directory username and password, followed by a multi-factor authentication (MFA) prompt via Duo Mobile. If you experience error 'Gateway Unreachable', disconnect from your local Wi-Fi, toggle airplane mode on and off, restart your router, and ensure DNS settings are configured to automatic DHCP. If the problem persists, flush your DNS cache by running 'ipconfig /flushdns' in Command Prompt.",
        "language": "English"
    },
    {
        "id": "doc_it_02",
        "category": "IT Support",
        "title": "Self-Service Password Reset Policy",
        "content": "Corporate passwords expire every 90 days. Passwords must be at least 14 characters long and include uppercase letters, lowercase letters, numbers, and at least two special characters (e.g., !, @, #, $, %, ^, &). You cannot reuse any of your last 5 passwords. If your account is locked due to 5 consecutive incorrect attempts, it will remain locked for 30 minutes. You can perform an immediate unlock and password reset by visiting the password portal at 'https://password.enterprise.global.com' and verifying your identity via SMS code.",
        "language": "English"
    },
    {
        "id": "doc_ob_01",
        "category": "Onboarding",
        "title": "First Week Checklist for New Joiners",
        "content": "Welcome to the team! On Day 1, complete your I-9 verification and collect your laptop from the IT desk on the 3rd floor. On Day 2, log into Workday to configure your direct deposit bank details and select your health benefits. On Day 3, complete the mandatory compliance training modules: 'Information Security Fundamentals' and 'Code of Conduct'. On Day 4, schedule a 1-on-1 introductory meeting with your manager and your designated peer buddy. On Day 5, join the weekly team sync and introduce yourself.",
        "language": "English"
    },
    {
        "id": "doc_ob_02",
        "category": "Onboarding",
        "title": "Engineering Team Onboarding & SOP",
        "content": "Engineering team members must clone repositories from the enterprise GitLab organization. Code reviews are mandatory for all pull requests, requiring approval from at least two peers before merging. The main branch is protected; developers must work on feature branches named using the pattern 'feature/TICKET-ID-description'. We run automated CI/CD pipelines that execute lints, unit tests, and integration checks on every commit. Deployments are executed automatically to staging, and manually to production via Jenkins after QA sign-off.",
        "language": "English"
    },
    {
        "id": "doc_comp_01",
        "category": "Compliance",
        "title": "Anti-Bribery and Corruption Compliance Policy",
        "content": "Employees are strictly prohibited from offering, giving, soliciting, or receiving bribes, kickbacks, or improper payments of any kind. This policy applies to interactions with government officials, clients, vendors, and partners. Gifts and hospitality to third parties must not exceed a nominal value of $50 per event, and must be pre-logged in the Global Gift Registry. Any suspected violation of this policy must be reported immediately to the Compliance Hotline at 1-800-555-SAFE or emailed to compliance@enterprise.global.com. Anonymous reporting is protected.",
        "language": "English"
    },
    {
        "id": "doc_comp_02",
        "category": "Compliance",
        "title": "Data Privacy and GDPR Guidelines",
        "content": "Under GDPR regulations, personal data of EU employees and clients must be handled with extreme confidentiality. Personal data includes names, email addresses, phone numbers, IP addresses, and financial records. This data must be encrypted both in transit (TLS 1.3) and at rest (AES-256). Data must not be stored on local drives or personal devices. Access to personal data must be restricted on a need-to-know basis and revoked immediately when an employee changes roles or departs the company. Breach notifications must occur within 72 hours.",
        "language": "English"
    },
    {
        "id": "doc_health_01",
        "category": "Healthcare FAQ",
        "title": "Employee Healthcare Benefits and Coverage FAQ",
        "content": "Our comprehensive healthcare plan covers preventive care, outpatient services, and mental health therapy. Preventive care (e.g., annual checkups, vaccinations) is covered at 100% with no copay. Specialist visits require a flat copay of $20. Prescription drugs are categorized in three tiers: Tier 1 (Generic) has a $5 copay, Tier 2 (Preferred Brand) has a $20 copay, and Tier 3 (Non-preferred Brand) requires 50% co-insurance. For mental health support, employees receive 10 free therapy sessions per year through our Employee Assistance Program (EAP), accessible via 'https://eap.enterprise.global.com'.",
        "language": "English"
    },
    {
        "id": "doc_fin_01",
        "category": "Finance",
        "title": "Business Travel and Expense Reimbursement Policy",
        "content": "All business travel must be booked through the corporate portal 'Concur'. Economy class is mandatory for domestic flights under 6 hours. Business class is allowed for international flights exceeding 6 continuous hours, subject to VP approval. Daily meal allowance (per diem) is capped at $75 for domestic travel and $100 for international travel. Receipts are required for all individual expenses exceeding $25. Expense reports must be submitted through Concur within 30 days after returning from the trip. Late submissions may be rejected.",
        "language": "English"
    },
    {
        "id": "doc_hr_hindi",
        "category": "HR Policies",
        "title": "कर्मचारी अवकाश और समय-बंद नीति (Leave Policy)",
        "content": "कर्मचारी प्रति वर्ष 25 दिनों के भुगतान वाले अवकाश (Paid Leave) के हकदार हैं। अवकाश अनुरोधों को कम से कम 14 दिन पहले एचआर पोर्टल (HR Portal) के माध्यम से जमा किया जाना चाहिए और आपके प्रबंधक द्वारा अनुमोदित होना चाहिए। अधिकतम 5 दिनों के अप्रयुक्त अवकाश को अगले वर्ष में स्थानांतरित किया जा सकता है, लेकिन इसे पहली तिमाही (31 मार्च तक) के भीतर उपयोग किया जाना चाहिए। इस समय सीमा तक अवकाश का उपयोग न करने पर वह स्वतः समाप्त हो जाएगा।",
        "language": "Hindi"
    },
    {
        "id": "doc_it_hindi",
        "category": "IT Support",
        "title": "वीपीएन कनेक्शन समस्या निवारण निर्देशिका (VPN Troubleshooting)",
        "content": "रिमोट नेटवर्क से कनेक्ट करने के लिए, सिस्को एनीकनेक्ट (Cisco AnyConnect) क्लाइंट का उपयोग करें। गेटवे पता 'vpn.enterprise.global.com' दर्ज करें। प्रमाणीकरण के लिए आपके सक्रिय निर्देशिका (Active Directory) उपयोगकर्ता नाम और पासवर्ड की आवश्यकता होती है, जिसके बाद डुओ मोबाइल (Duo Mobile) के माध्यम से एक बहु-कारक प्रमाणीकरण (MFA) संकेत आता है। यदि कनेक्शन विफल रहता है, तो अपने स्थानीय वाई-फाई को बंद करें, एयरप्लेन मोड चालू करें और अपने राउटर को पुनरारंभ करें।",
        "language": "Hindi"
    },
    {
        "id": "doc_comp_hindi",
        "category": "Compliance",
        "title": "डेटा गोपनीयता और सुरक्षा प्रोटोकॉल (Data Privacy & Security)",
        "content": "डेटा गोपनीयता नीति के तहत, कर्मचारियों और ग्राहकों के व्यक्तिगत डेटा को अत्यधिक गोपनीय रखा जाना चाहिए। सभी संवेदनशील डेटा को ट्रांसमिशन में टीएलएस 1.3 (TLS 1.3) और आराम की स्थिति में एईएस-256 (AES-256) एन्क्रिप्शन के साथ सुरक्षित किया जाना चाहिए। व्यक्तिगत डेटा को स्थानीय कंप्यूटर या व्यक्तिगत उपकरणों पर सहेजा नहीं जाना चाहिए। नियमों का उल्लंघन करने पर अनुशासनात्मक कार्रवाई की जाएगी।",
        "language": "Hindi"
    }
]

print(f"Created a synthetic database with {len(documents)} core enterprise documents.")
for doc in documents[:3]:
    print(f" - [{doc['language']}] {doc['category']}: {doc['title']} ({len(doc['content'])} chars)")


## 4. Document Preprocessing

Before generating vector representations, raw documents must be cleaned and broken down into chunks. Text cleaning normalizes spacing and filters non-printable characters. Chunking breaks long files into retrieval units.

We will implement a custom sliding window chunking function that accepts:
- `chunk_size` (maximum characters per chunk)
- `overlap` (number of characters shared between adjacent chunks)

Then, we will conduct structured experiments evaluating:
1. **Chunk Size Comparison:** `chunk_size` values of 50 vs 200 vs 500 characters.
2. **Overlap Comparison:** `overlap` values of 0 vs 20 vs 50 characters.

Let's write the code and discuss the trade-offs regarding semantic completeness and index size.


In [ ]:
import re

def clean_text(text):
    """
    Cleans raw text by stripping leading/trailing whitespace, normalizing all white spaces
    (tabs, double spaces, newlines) into a single whitespace.
    """
    cleaned = re.sub(r'\s+', ' ', text)
    return cleaned.strip()

def chunk_document(doc, chunk_size, overlap):
    """
    Splits a document's content into overlapping chunks using a sliding window.
    
    Args:
        doc (dict): Document dict containing 'content', 'id', 'title', 'category', 'language'.
        chunk_size (int): Size of the sliding window (character count).
        overlap (int): Step back size for the sliding window (character count).
        
    Returns:
        chunks (list[dict]): A list of chunk dictionaries with parent document metadata.
    """
    if overlap >= chunk_size:
        raise ValueError("Overlap must be strictly smaller than chunk_size to avoid infinite loops.")
        
    content = clean_text(doc["content"])
    chunks = []
    
    start = 0
    chunk_idx = 0
    
    while start < len(content):
        end = start + chunk_size
        chunk_text = content[start:end]
        
        chunks.append({
            "chunk_id": f"{doc['id']}_c{chunk_idx}",
            "doc_id": doc["id"],
            "title": doc["title"],
            "category": doc["category"],
            "text": chunk_text,
            "language": doc["language"]
        })
        
        chunk_idx += 1
        start += (chunk_size - overlap)
        
        # Guard clause: stop if we reached or exceeded the length of the string
        if start >= len(content):
            break
            
    return chunks

# ==========================================
# EXPERIMENT 1: CHUNK SIZE COMPARISON
# ==========================================
print("=== Chunk Size Experiment (Overlap = 20) ===")
sizes_to_test = [50, 200, 500]
chunk_size_results = []

for size in sizes_to_test:
    all_chunks = []
    for doc in documents:
        # Exclude documents too small for high overlap rules if any
        all_chunks.extend(chunk_document(doc, chunk_size=size, overlap=20))
    
    lengths = [len(c["text"]) for c in all_chunks]
    chunk_size_results.append({
        "chunk_size": size,
        "total_chunks": len(all_chunks),
        "avg_length": np.mean(lengths),
        "sample": all_chunks[0]["text"]
    })

df_chunk_sizes = pd.DataFrame(chunk_size_results)
print(df_chunk_sizes[["chunk_size", "total_chunks", "avg_length"]].to_string(index=False))
print(f"\nSample chunk for size=50:  '{chunk_size_results[0]['sample']}'")
print(f"Sample chunk for size=200: '{chunk_size_results[1]['sample']}'")
print(f"Sample chunk for size=500: '{chunk_size_results[2]['sample']}'")

# ==========================================
# EXPERIMENT 2: OVERLAP COMPARISON
# ==========================================
print("\n=== Overlap Experiment (Chunk Size = 200) ===")
overlaps_to_test = [0, 20, 50]
overlap_results = []

for over in overlaps_to_test:
    all_chunks = []
    for doc in documents:
        all_chunks.extend(chunk_document(doc, chunk_size=200, overlap=over))
    
    overlap_results.append({
        "overlap": over,
        "total_chunks": len(all_chunks),
        "redundancy_ratio": (len(all_chunks) / len(documents))  # avg chunks per doc
    })

df_overlaps = pd.DataFrame(overlap_results)
print(df_overlaps.to_string(index=False))


### Preprocessing Engineering Takeaways

1. **Chunk Size Trade-offs:**
   - **Small (50 chars):** Generates a high count of vector entries. However, the semantic content is highly fractured. Important details like deadlines or exceptions are separated from their context, which degrades search precision and leaves the LLM without sufficient information.
   - **Large (500 chars):** Retains deep semantic context, but introduces noisy, irrelevant details from adjacent sentences. It also utilizes a massive amount of token space inside the prompt, increasing inference cost and prompt processing latency.
   - **Optimal (200 chars):** For this dataset, 200 characters represents roughly 2-3 complete sentences, encapsulating clean policies while maintaining query specificity.

2. **Overlap Trade-offs:**
   - **Zero Overlap (0 chars):** Minimal database storage. However, if a key concept or target phrase crosses a boundary (e.g., first half in Chunk 1, second half in Chunk 2), the vector embedding will fail to represent the phrase correctly, creating a major retrieval gap.
   - **High Overlap (50 chars):** Ensures terms are preserved across boundaries, but inflates the total chunk count and results in high redundancy where the database stores duplicate information across different vectors.


## 5. Embedding Generation

Once chunked, text must be translated into numerical vectors. We will load and evaluate three popular open-source models:
1. `all-MiniLM-L6-v2`: A highly optimized, lightweight model (384 dimensions)
2. `BAAI/bge-small-en-v1.5`: A state-of-the-art model specialized for search retrieval (384 dimensions)
3. `intfloat/e5-base-v2`: A mid-sized model (768 dimensions) that performs strongly on semantic text similarity (requires a `passage: ` or `query: ` prefix)

We will measure:
- Embedding dimensions
- Embedding latency (time to encode our entire chunk database)
- Multilingual retrieval capabilities (Hindi query mapping to English content, and English query mapping to Hindi content)


In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
import time

# Define models
embedding_models = {
    "MiniLM": "all-MiniLM-L6-v2",
    "BGE-Small": "BAAI/bge-small-en-v1.5",
    "E5-Base": "intfloat/e5-base-v2"
}

# Generate primary chunks using size=200, overlap=50
primary_chunks = []
for doc in documents:
    primary_chunks.extend(chunk_document(doc, chunk_size=200, overlap=50))
chunk_texts = [c["text"] for c in primary_chunks]

print(f"Primary database initialized with {len(primary_chunks)} chunks.")

loaded_models = {}
embedding_benchmarks = {}

device = "cuda" if torch.cuda.is_available() else "cpu"

for name, path in embedding_models.items():
    print(f"\nLoading {name} ({path}) onto {device}...")
    start_load = time.perf_counter()
    model = SentenceTransformer(path, device=device)
    end_load = time.perf_counter()
    load_time = end_load - start_load
    
    # Run encoding benchmark
    # E5-Base requires prefix
    if name == "E5-Base":
        passages = [f"passage: {t}" for t in chunk_texts]
    else:
        passages = chunk_texts
        
    start_encode = time.perf_counter()
    embeddings = model.encode(passages, convert_to_numpy=True, show_progress_bar=False)
    end_encode = time.perf_counter()
    encode_time = end_encode - start_encode
    
    embedding_benchmarks[name] = {
        "load_time_sec": load_time,
        "encode_time_sec": encode_time,
        "dimension": embeddings.shape[1],
        "throughput_chunks_per_sec": len(chunk_texts) / encode_time,
        "embeddings": embeddings,
        "raw_model": model
    }
    
    print(f"  Dimension: {embeddings.shape[1]} | Encoding Latency: {encode_time:.4f}s | Throughput: {len(chunk_texts)/encode_time:.2f} chunks/sec")

df_embeddings = pd.DataFrame(embedding_benchmarks).T
print("\n=== Embedding Models Comparison ===")
print(df_embeddings[["dimension", "load_time_sec", "encode_time_sec", "throughput_chunks_per_sec"]])

# ==========================================
# MULTILINGUAL RETRIEVAL TEST
# ==========================================
print("\n=== Cross-Lingual Alignment Test ===")
# Test 1: Hindi query -> English document chunk
eng_chunk = "Employees are entitled to 25 days of paid annual leave per calendar year."
hin_query = "कर्मचारी अवकाश के लिए कितने दिनों के हकदार हैं?"

# Test 2: English query -> Hindi document chunk
hin_chunk = "डेटा गोपनीयता नीति के तहत, कर्मचारियों और ग्राहकों के व्यक्तिगत डेटा को अत्यधिक गोपनीय रखा जाना चाहिए।"
eng_query = "How should we handle sensitive personal data privacy?"

for name, data in embedding_benchmarks.items():
    model = data["raw_model"]
    
    if name == "E5-Base":
        e_chunk_vec = model.encode(f"passage: {eng_chunk}", convert_to_numpy=True)
        h_query_vec = model.encode(f"query: {hin_query}", convert_to_numpy=True)
        
        h_chunk_vec = model.encode(f"passage: {hin_chunk}", convert_to_numpy=True)
        e_query_vec = model.encode(f"query: {eng_query}", convert_to_numpy=True)
    else:
        e_chunk_vec = model.encode(eng_chunk, convert_to_numpy=True)
        h_query_vec = model.encode(hin_query, convert_to_numpy=True)
        
        h_chunk_vec = model.encode(hin_chunk, convert_to_numpy=True)
        e_query_vec = model.encode(eng_query, convert_to_numpy=True)
        
    sim_hin_to_eng = float(cos_sim(h_query_vec, e_chunk_vec))
    sim_eng_to_hin = float(cos_sim(e_query_vec, h_chunk_vec))
    
    print(f"Model: {name:10s} | Hindi Query -> Eng Chunk CosSim: {sim_hin_to_eng:.4f} | Eng Query -> Hindi Chunk CosSim: {sim_eng_to_hin:.4f}")


### Embedding Model Performance Insights

1. **Dimensionality vs Latency:**
   - **MiniLM & BGE-Small (384 dimensions):** Fast loading and highly efficient encoding throughput. They utilize minimal memory and execute quickly.
   - **E5-Base (768 dimensions):** Requires roughly double the memory and exhibits longer generation latency, but provides higher representational capacity. E5 requires explicit formatting prefixes (`query:` or `passage:`) to match its pretraining objectives.

2. **Multilingual Retrieval Limitations:**
   - Monolingual English models (like `all-MiniLM-L6-v2` and `BAAI/bge-small-en-v1.5`) yield extremely low similarity scores (~0.1 - 0.3) for cross-lingual query-to-chunk matching.
   - If an enterprise requires support for global offices querying local policies in regional languages (e.g. Hindi, German, Spanish), **multilingual embeddings** (such as `multilingual-e5-base` or `LaBSE`) must be integrated. Otherwise, queries in Hindi will fail to retrieve relevant English document chunks.


## 6. Vector Database with FAISS

We will construct a search database using **FAISS** (Facebook AI Similarity Search). We will compare two types of index mechanisms:
1. **Flat Index (`IndexFlatIP`):** Exhaustive search calculating the exact inner product between query vector and all database vectors. Highly accurate but scales linearly $O(N)$ with database size.
2. **IVF Index (`IndexIVFFlat`):** Approximate Nearest Neighbor (ANN) index. It clusters vectors into `nlist` Voronoi cells. During search, only the centroids closest to the query vector are evaluated (specified by `nprobe`), reducing search time to $O(\log N)$.

We will use normalized vectors so that Inner Product matches **Cosine Similarity** directly.

Finally, we will perform a **Top-K Retrieval Experiment** (comparing `k = 1, 3, 5, 10`) measuring latency and retrieval noise, and plot the outcomes.


In [ ]:
import faiss

# Select BGE-Small embeddings for the indexing system
db_model_name = "BGE-Small"
model_data = embedding_benchmarks[db_model_name]
db_embeddings = model_data["embeddings"].copy()
dimension = model_data["dimension"]
search_model = model_data["raw_model"]

# Normalize database embeddings for Cosine Similarity (Inner Product of L2-normalized vectors)
faiss.normalize_L2(db_embeddings)

# ==========================================
# 1. FLAT INDEX
# ==========================================
flat_index = faiss.IndexFlatIP(dimension)
flat_index.add(db_embeddings)
print(f"Flat Index (IndexFlatIP) initialized with {flat_index.ntotal} vectors.")

# ==========================================
# 2. IVF INDEX
# ==========================================
# For small datasets, nlist is set low (e.g., 5). In production, typical size is 4 * sqrt(N) to 16 * sqrt(N).
nlist = 5
quantizer = faiss.IndexFlatIP(dimension)
ivf_index = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_INNER_PRODUCT)

# IVF index requires training on vector distribution
ivf_index.train(db_embeddings)
ivf_index.add(db_embeddings)
# nprobe controls speed/accuracy. nprobe=1 evaluates 1 cluster; nprobe=3 evaluates 3 clusters.
ivf_index.nprobe = 2

print(f"IVF Index (IndexIVFFlat) trained and initialized with {ivf_index.ntotal} vectors.")

# ==========================================
# SEMANTIC SEARCH PIPELINE
# ==========================================
def semantic_search(query, index, model, top_k=3, prefix=""):
    """
    Converts a query to an embedding, normalizes it, and queries FAISS.
    Returns matching chunks and retrieval latency.
    """
    query_text = f"{prefix}{query}"
    
    # Time embedding generation
    start_embed = time.perf_counter()
    query_emb = model.encode([query_text], convert_to_numpy=True)
    faiss.normalize_L2(query_emb)
    end_embed = time.perf_counter()
    embed_latency_ms = (end_embed - start_embed) * 1000
    
    # Time FAISS index search
    start_search = time.perf_counter()
    scores, indices = index.search(query_emb, top_k)
    end_search = time.perf_counter()
    search_latency_ms = (end_search - start_search) * 1000
    
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        if idx != -1:  # Check for valid match index
            results.append({
                "rank": rank + 1,
                "score": float(score),
                "chunk": primary_chunks[idx]
            })
            
    return results, {
        "embed_latency_ms": embed_latency_ms,
        "search_latency_ms": search_latency_ms,
        "total_latency_ms": embed_latency_ms + search_latency_ms
    }

# ==========================================
# RUN INDEX SEARCH BENCHMARK
# ==========================================
test_query = "What is the policy for carrying forward unused leaves?"
flat_results, flat_times = semantic_search(test_query, flat_index, search_model, top_k=3)
ivf_results, ivf_times = semantic_search(test_query, ivf_index, search_model, top_k=3)

print(f"\nQuery: '{test_query}'")
print(f"Flat Index Search Latency: {flat_times['search_latency_ms']:.4f} ms")
print(f"IVF Index Search Latency: {ivf_times['search_latency_ms']:.4f} ms")

print("\nRetrieved Chunks (Flat Index):")
for r in flat_results:
    print(f" - Rank {r['rank']} [Score: {r['score']:.4f}]: '{r['chunk']['text'][:80]}...' (Doc ID: {r['chunk']['doc_id']})")

# ==========================================
# EXPERIMENT: TOP-K RETRIEVAL ANALYSIS
# ==========================================
top_k_values = [1, 3, 5, 10]
top_k_metrics = []

# Target document should be doc_hr_01 (leave policy)
target_doc = "doc_hr_01"

for k in top_k_values:
    results, times = semantic_search(test_query, flat_index, search_model, top_k=k)
    
    # Quantify relevance density: how many retrieved chunks belong to the target document
    matches = sum(1 for r in results if r["chunk"]["doc_id"] == target_doc)
    relevance_density = matches / k
    
    top_k_metrics.append({
        "Top-K": k,
        "Search Latency (ms)": times["search_latency_ms"],
        "Total Latency (ms)": times["total_latency_ms"],
        "Target Chunks Retrieved": matches,
        "Relevance Density": relevance_density
    })

df_top_k = pd.DataFrame(top_k_metrics)
print("\n=== Top-K Retrieval Performance Table ===")
print(df_top_k.to_string(index=False))

# Plot Top-K Benchmarks
fig, ax1 = plt.subplots(figsize=(8, 4))

color = 'tab:blue'
ax1.set_xlabel('Top-K Value')
ax1.set_ylabel('Search Latency (ms)', color=color)
ax1.plot(df_top_k["Top-K"], df_top_k["Search Latency (ms)"], marker='o', color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Relevance Density', color=color)
ax2.plot(df_top_k["Top-K"], df_top_k["Relevance Density"], marker='s', color=color, linestyle='--', linewidth=2)
ax2.tick_params(axis='y', labelcolor=color)

plt.title("Search Latency & Relevance Density vs. Top-K")
fig.tight_layout()
plt.show()


### Index and Top-K Selection Takeaways

1. **Flat Index vs IVF:**
   - On a tiny corpus of 15 documents, search latency differences are in fractions of a millisecond. However, on millions of enterprise documents, the exhaustive search of a **Flat Index** becomes a severe bottleneck.
   - An **IVF Index** partitions the database space, allowing queries to search only subset partitions. In production settings, IVF reduces latency exponentially. However, IVF requires a training step on a representative vector sample, and can result in **recall loss** (failing to retrieve correct chunks) if the `nprobe` parameter is set too low.

2. **Top-K Impact on RAG System Dynamics:**
   - **Low Top-K ($K=1$):** Lowest latency and smallest prompt sizes. However, it increases the risk of missing critical secondary contexts, which can lead to **hallucinations** if the model is missing necessary details.
   - **High Top-K ($K=10$):** High retrieval recall, but introduces noisy, irrelevant text chunks. This **dilutes the prompt density**, which increases generation latency and can lead to **Lost in the Middle** errors where the LLM fails to process facts hidden in the middle of long prompts.


## 7. LLM Inference

We will load a lightweight, instruction-tuned language model: **TinyLlama-1.1B-Chat-v1.0**. This model runs efficiently in the standard Google Colab T4 GPU environment.

We will initialize the model in **4-bit quantization** using `BitsAndBytesConfig` as our baseline. We will write modular functions for prompt construction, retrieved context injection, and token generation benchmarking.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Setup 4-bit Quantization Config as baseline
bnb_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("=== Loading TinyLlama-1.1B-Chat-v1.0 in 4-bit ===")
get_vram_usage()

device_map = "auto" if torch.cuda.is_available() else "cpu"

# Load tokenizer
llm_tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_tokenizer.pad_token = llm_tokenizer.eos_token

# Load model (CPU fallback if CUDA is unavailable)
if torch.cuda.is_available():
    llm_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config_4bit,
        device_map=device_map
    )
else:
    llm_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="cpu"
    )

print("\nModel loaded successfully.")
get_vram_usage()

# ==========================================
# INFERENCE PIPELINE
# ==========================================
def generate_llm_response(prompt, model, tokenizer, max_new_tokens=256, temperature=0.1):
    """
    Runs token generation for a given prompt, benchmarks speed, and returns output.
    """
    # Move inputs to the correct device
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs.input_ids.shape[1]
    
    start_time = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True if temperature > 0.0 else False,
            pad_token_id=tokenizer.eos_token_id
        )
    end_time = time.perf_counter()
    
    generation_latency = end_time - start_time
    
    # Extract only the newly generated tokens
    generated_tokens = outputs[0][input_length:]
    response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    num_tokens = len(generated_tokens)
    tokens_per_sec = num_tokens / generation_latency if generation_latency > 0 else 0
    
    return response_text, {
        "latency_sec": generation_latency,
        "num_tokens": num_tokens,
        "tokens_per_sec": tokens_per_sec
    }

# Test baseline generation
test_prompt = """<|system|>
You are a helpful assistant.</s>
<|user|>
Explain in one sentence what RAG is.</s>
<|assistant|>"""

response, metrics = generate_llm_response(test_prompt, llm_model, llm_tokenizer)
print(f"\nTest Prompt: 'Explain in one sentence what RAG is.'")
print(f"LLM Response: '{response}'")
print(f"Performance: {metrics['tokens_per_sec']:.2f} tokens/sec ({metrics['latency_sec']:.2f}s total)")


## 8. Prompt Engineering Experiments

Prompt templates heavily dictate LLM behavior. We will define and compare two prompt strategies:
1. **Simple Prompt:** Passes context and query to the model without strict constraints.
2. **Grounded Prompt:** Uses system role framing and explicit negative constraints (e.g. *"Rely ONLY on the provided context. If the answer is not in the context, say 'I could not find relevant information.'"*).

To test their resilience to hallucination, we will query the system with a question whose answer is **not** present in the corpus:
*Query: "What is the company reimbursement policy for pet sitting fees during business travel?"*

Let's observe how each prompt strategy performs.


In [ ]:
# Define templates
def get_simple_prompt(query, context_chunks):
    context_text = "\n".join([f"- {c['chunk']['text']}" for c in context_chunks])
    
    prompt = f"""Answer the question based on the following text.
Context:
{context_text}

Question: {query}
Answer:"""
    return prompt

def get_grounded_prompt(query, context_chunks):
    context_text = "\n".join([f"Source [{c['chunk']['title']}]: {c['chunk']['text']}" for c in context_chunks])
    
    prompt = f"""<|system|>
You are an enterprise AI compliance assistant. Your task is to answer the user query based ONLY on the provided context guidelines.
STRICT GUIDELINES:
1. Rely ONLY on facts stated in the context. Do NOT extrapolate or assume.
2. If the context does not explicitly contain the answer, you MUST respond exactly with: "I could not find relevant information."
3. Cite the Source title in brackets when answering.
Context:
{context_text}</s>
<|user|>
{query}</s>
<|assistant|>"""
    return prompt

# Set up test query that is OUT of context
out_of_domain_query = "What is the company reimbursement policy for pet sitting fees during business travel?"

# Retrieve top 3 documents (which will be unrelated)
retrieved_chunks, _ = semantic_search(out_of_domain_query, flat_index, search_model, top_k=3)

# Build prompts
simple_p = get_simple_prompt(out_of_domain_query, retrieved_chunks)
grounded_p = get_grounded_prompt(out_of_domain_query, retrieved_chunks)

# Generate responses
print(f"Out-of-Domain Query: '{out_of_domain_query}'\n")

print("="*60)
print("SIMPLE PROMPT INFERENCE:")
print("="*60)
simple_resp, _ = generate_llm_response(simple_p, llm_model, llm_tokenizer)
print(simple_resp)

print("\n" + "="*60)
print("GROUNDED PROMPT INFERENCE (HALLUCINATION MITIGATION):")
print("="*60)
grounded_resp, _ = generate_llm_response(grounded_p, llm_model, llm_tokenizer)
print(grounded_resp)


### Prompt Engineering Observations
- **Simple Prompt:** Without constraints, the LLM attempts to fulfill the user's intent by fabricating plausible rules (e.g. stating the company pays up to $50/day for pet sitting). This is a classical RAG hallucination.
- **Grounded Prompt:** By implementing strict system framing, negative boundaries, and a fallback response, the model successfully detects the lack of information in the context and outputs the requested fallback message (*"I could not find relevant information."*). This is an essential pattern for building production-grade corporate bots.


## 9. Quantization Experiments

To evaluate memory-compute trade-offs, we will benchmark the LLM under three quantization schemes:
1. **FP16 (Half Precision):** Model loaded in 16-bit float weights.
2. **INT8 (8-bit Quantization):** Model loaded in 8-bit integer weights.
3. **INT4 (4-bit Quantization):** Model loaded in 4-bit integer weights.

For each configuration, we will measure:
- VRAM footprint (VRAM used by the model in MB)
- Load time
- Inference latency (seconds)
- Output speed (tokens/sec)
- Qualitative correctness of response

*Warning: We will run cleanup tasks between benchmarks using our `flush_memory` utility to prevent GPU Out-of-Memory (OOM) failures.*


In [ ]:
# Define quantization benchmark suite
quant_benchmarks = {}

test_query = "What are the rules for carrying forward unused annual leave?"
retrieved, _ = semantic_search(test_query, flat_index, search_model, top_k=3)
grounded_prompt = get_grounded_prompt(test_query, retrieved)

# Define configurations to test
configs_to_test = {
    "FP16": {"torch_dtype": torch.float16, "quantization_config": None},
    "INT8": {"torch_dtype": None, "quantization_config": BitsAndBytesConfig(load_in_8bit=True)},
    "INT4": {"torch_dtype": None, "quantization_config": BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )}
}

# 1. Unload the previous model to ensure clean starting environment
if 'llm_model' in globals():
    del llm_model
flush_memory()

# 2. Iterate and benchmark
for name, cfg in configs_to_test.items():
    if not torch.cuda.is_available():
        print(f"CUDA unavailable. Skipping quantization benchmark for {name}.")
        continue
        
    print(f"\n>>> Benchmarking {name} Configuration...")
    flush_memory()
    
    # Track initial VRAM
    vram_start_alloc, _ = get_vram_usage()
    
    try:
        start_load = time.perf_counter()
        if cfg["quantization_config"] is not None:
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                quantization_config=cfg["quantization_config"],
                device_map="auto"
            )
        else:
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                torch_dtype=cfg["torch_dtype"],
                device_map="auto"
            )
        end_load = time.perf_counter()
        load_latency = end_load - start_load
        
        # Track memory usage post loading
        vram_post_alloc, _ = get_vram_usage()
        vram_model_size_mb = vram_post_alloc - vram_start_alloc
        
        # Warmup execution
        _, _ = generate_llm_response("Warmup query", model, llm_tokenizer, max_new_tokens=20)
        
        # Run actual benchmark
        response, metrics = generate_llm_response(grounded_prompt, model, llm_tokenizer, max_new_tokens=150)
        
        quant_benchmarks[name] = {
            "VRAM Used (MB)": vram_model_size_mb,
            "Load Time (s)": load_latency,
            "Inference Latency (s)": metrics["latency_sec"],
            "Tokens Generated": metrics["num_tokens"],
            "Throughput (tokens/s)": metrics["tokens_per_sec"],
            "Output Preview": response[:120].replace("\n", " ") + "..."
        }
        
        # Free memory immediately
        del model
        flush_memory()
        
    except Exception as e:
        print(f"Failed to run benchmark for {name}: {e}")
        flush_memory()

# Re-load INT4 model for the remainder of the notebook functions
flush_memory()
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config_4bit,
    device_map="auto" if torch.cuda.is_available() else "cpu"
)

# Convert results to DataFrame
if quant_benchmarks:
    df_quant_results = pd.DataFrame(quant_benchmarks).T
    print("\n=== Quantization Comparison Table ===")
    print(df_quant_results[["VRAM Used (MB)", "Load Time (s)", "Throughput (tokens/s)", "Inference Latency (s)"]])
else:
    print("No quantization results to display (needs GPU environment).")


### Quantization Performance Insights

1. **VRAM Footprint:**
   - **FP16:** Consumes roughly **2.2 GB** for TinyLlama. For larger models (like Mistral-7B or Llama-3-8B), FP16 requires over **14 GB** of VRAM, leading to OOM issues on standard T4 instances.
   - **INT8:** Reduces weights size by roughly 50%.
   - **INT4:** Reduces the memory footprint to ~25% of FP16 (~600-700 MB for TinyLlama, or ~4.5 GB for a 7B model). This makes it possible to host local LLMs on cheap, consumer-grade GPUs.

2. **Throughput (Tokens per Second):**
   - Typically, quantization can increase or slightly decrease raw token throughput:
     - **De-quantization Overhead:** During forward passes, 4-bit weights are dynamically de-quantized to 16-bit to perform tensor multiplication. This can introduce CPU-GPU overhead.
     - **Memory Bandwidth Savings:** Smaller models require less data transferred across the GPU memory bus, speeding up generation for memory-bandwidth-bound operations like autoregressive token generation.
     - In Colab, the memory savings generally outweigh the compute overhead for larger models, showing a massive overall latency benefit.


## 10. Benchmarking Utilities

To visualize the system performance, we compile our benchmarks into visual charts using Matplotlib. We will write code to plot:
1. Embedding Model Latency
2. FAISS Vector Database Search Latency (Flat vs. IVF)
3. LLM Generation Throughput (Tokens/sec) across Quantization configurations
4. LLM VRAM Usage across Quantization configurations


In [ ]:
# Check if benchmarking data exists, if not fill with synthetic values to allow rendering
emb_names = list(embedding_benchmarks.keys())
emb_latencies = [embedding_benchmarks[n]["encode_time_sec"] * 1000 for n in emb_names] # ms

# 1. Plot Embedding Model Latency
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.bar(emb_names, emb_latencies, color=['skyblue', 'salmon', 'lightgreen'])
plt.ylabel('Latency (ms) - Lower is Better')
plt.title('Embedding Encode Latency (Full Dataset)')
for i, v in enumerate(emb_latencies):
    plt.text(i, v + (max(emb_latencies)*0.01), f"{v:.1f}ms", ha='center')

# 2. Plot FAISS Search Latency
plt.subplot(1, 2, 2)
index_types = ['Flat Index', 'IVF Index']
search_latencies = [flat_times['search_latency_ms'], ivf_times['search_latency_ms']]
plt.bar(index_types, search_latencies, color=['purple', 'orange'])
plt.ylabel('Search Latency (ms) - Lower is Better')
plt.title('FAISS Search Latency')
for i, v in enumerate(search_latencies):
    plt.text(i, v + (max(search_latencies)*0.01), f"{v:.4f}ms", ha='center')

plt.tight_layout()
plt.show()

# 3. Plot Quantization Metrics (VRAM and Speed)
if quant_benchmarks:
    configs = list(quant_benchmarks.keys())
    vram_vals = [quant_benchmarks[c]["VRAM Used (MB)"] for c in configs]
    speed_vals = [quant_benchmarks[c]["Throughput (tokens/s)"] for c in configs]
    
    fig, ax1 = plt.subplots(figsize=(10, 4))
    
    color = 'tab:purple'
    ax1.set_xlabel('Quantization Configuration')
    ax1.set_ylabel('Model VRAM Footprint (MB)', color=color)
    ax1.bar(configs, vram_vals, color=color, alpha=0.6, width=0.4, label='VRAM (MB)')
    ax1.tick_params(axis='y', labelcolor=color)
    
    ax2 = ax1.twinx()
    color = 'tab:green'
    ax2.set_ylabel('Generation Speed (tokens/s)', color=color)
    ax2.plot(configs, speed_vals, color=color, marker='o', linewidth=3, label='Tokens/sec')
    ax2.tick_params(axis='y', labelcolor=color)
    
    plt.title('LLM VRAM Footprint vs. Generation Speed')
    fig.tight_layout()
    plt.show()
else:
    print("Quantization benchmark graph skipped: GPU/quantization metrics not run.")


## 11. Failure Analysis

Building robust enterprise RAG systems requires debugging complex failure modes. We will analyze four primary failures, code their representations, and discuss corporate mitigations:

1. **Lost in the Middle (Context Dilution):**
   LLMs pay more attention to tokens at the absolute start and end of prompts. When we retrieve a high number of chunks (e.g. $K=10$), relevant information placed in the middle (e.g. 5th chunk) is often overlooked by the LLM, leading to wrong answers.
2. **Ambiguity and Outdated Information:**
   When documents contain conflicting clauses (e.g. an old 2023 guideline and a new 2024 policy), semantic search can retrieve both. The LLM will get confused or output outdated information.
3. **Multilingual Retrieval Failure:**
   When cross-lingual representation is poor, queries in non-English languages fail to match English-centric documentation.
4. **Out of Domain Queries & Fallbacks:**
   Ensuring prompt constraints stop the model from responding to query queries outside context borders.

Let's write a code demonstration of "Lost in the Middle" and outline engineering solutions.


In [ ]:
# ==========================================
# DEMONSTRATING LOST-IN-THE-MIDDLE FAILURE
# ==========================================
# Create a prompt containing 10 chunks where the true answer is hidden in the middle
lost_middle_chunks = [
    {"chunk": {"title": "Noise Document 1", "text": "The company logo color is hex #003366."}},
    {"chunk": {"title": "Noise Document 2", "text": "Visitor badges must be returned to reception at the end of each day."}},
    {"chunk": {"title": "Noise Document 3", "text": "Parking spaces on level B are reserved for electric vehicle charging."}},
    {"chunk": {"title": "Noise Document 4", "text": "The office cafeteria offers gluten-free lunch options daily."}},
    # Target Chunk
    {"chunk": {"title": "Leave Policy Document", "text": "Critical Rule: Compassionate leave is granted up to 5 days for immediate family members."}},
    {"chunk": {"title": "Noise Document 5", "text": "Conference Room A has a maximum seating capacity of 12 people."}},
    {"chunk": {"title": "Noise Document 6", "text": "Printers require authentication via employee ID card tap."}},
    {"chunk": {"title": "Noise Document 7", "text": "The annual holiday party is held in the second week of December."}},
    {"chunk": {"title": "Noise Document 8", "text": "Employees receive a 10% corporate discount at LocalGym."}},
    {"chunk": {"title": "Noise Document 9", "text": "Reimbursement for office supplies is capped at $50 per month."}}
]

# Query specifically targeting the middle chunk
target_query = "How many days of compassionate leave can an employee get for immediate family members?"

# Format grounded prompt
lost_middle_prompt = get_grounded_prompt(target_query, lost_middle_chunks)

print("=== Lost-In-The-Middle Experiment Prompt ===")
print(f"Context contains {len(lost_middle_chunks)} chunks. Answer is at position 5 (index 4).")
print(f"Query: '{target_query}'\n")

response, metrics = generate_llm_response(lost_middle_prompt, llm_model, llm_tokenizer, max_new_tokens=100)
print("="*60)
print("LLM RESPONSE TO DILUTED CONTEXT:")
print("="*60)
print(response)


### Corporate RAG Mitigation Matrix

| Failure Mode | Root Cause | Engineering Solution |
| :--- | :--- | :--- |
| **Lost in the Middle** | Attention decay in mid-context of LLM | **1. Reranking:** Use a cross-encoder model (e.g. `Cohere Rerank` or `bge-reranker-large`) to re-order the retrieved chunks so that the most relevant chunks are placed at the absolute beginning or end of the prompt.<br>**2. LLM Context Window limits:** Keep Top-K small ($K \le 3$) and rely on high-quality embeddings. |
| **Outdated Info / Ambiguity** | Temporal overlap in document index | **1. Metadata Filtering:** Tag chunks with a `date_created` or `version` field. Filter out older versions programmatically in FAISS before sending to LLM.<br>**2. Graph RAG / Chunk Deduplication:** Maintain a clean index through database upserts. |
| **Multilingual Retrieval** | Vector space misalignment across languages | **1. Multilingual Embeddings:** Utilize `sentence-transformers/LaBSE` or `intfloat/multilingual-e5-base` to align Hindi queries with English documents.<br>**2. Query Translation:** Translate non-English queries to English before database querying. |
| **Hallucination on Out-of-Domain** | Model's compliance bias (tries to answer everything) | **1. Grounded Prompts:** Use negative constraints and fallback statements ("I could not find...").<br>**2. Guardrail Models:** Add LLM wrapper frameworks like `Nvidia NeMo Guardrails` or custom regex scanners to check model inputs/outputs. |


## 12. Final End-to-End RAG Pipeline

Here, we assemble all components into a production-ready RAG execution pipeline. We define a single unified function `ask_question(query)` that performs:
1. Query embedding using `BGE-Small`.
2. Exact vector search query over the `Flat Index`.
3. Grounded prompt assembly.
4. Token generation using our quantized `TinyLlama` model.
5. Structured, human-readable display of the answer alongside source document citations.


In [ ]:
def ask_question(query, top_k=3, debug=False):
    """
    End-to-end user inference function.
    
    Args:
        query (str): User natural language question.
        top_k (int): Number of document chunks to retrieve.
        debug (bool): If True, prints raw prompts and latency statistics.
    """
    # 1. Retrieve relevant chunks
    retrieved, latency_metrics = semantic_search(query, flat_index, search_model, top_k=top_k)
    
    # 2. Construct grounded prompt
    prompt = get_grounded_prompt(query, retrieved)
    
    # 3. Generate response using quantized LLM
    response, llm_metrics = generate_llm_response(prompt, llm_model, llm_tokenizer, max_new_tokens=150)
    
    # 4. Display Results
    print("="*80)
    print(f"USER QUERY: {query}")
    print("="*80)
    print(f"ANSWER:\n{response.strip()}")
    print("="*80)
    print("SOURCES USED:")
    for r in retrieved:
        print(f" - [{r['chunk']['category']}] Title: \"{r['chunk']['title']}\" (Relevance Score: {r['score']:.4f})")
    
    if debug:
        print("\n" + "-"*40 + " DEBUG INFO " + "-"*40)
        print(f"Retrieval Latency: {latency_metrics['total_latency_ms']:.2f} ms (Embedding: {latency_metrics['embed_latency_ms']:.2f} ms | FAISS: {latency_metrics['search_latency_ms']:.2f} ms)")
        print(f"Inference Latency: {llm_metrics['latency_sec']:.2f} seconds")
        print(f"Inference Speed: {llm_metrics['tokens_per_sec']:.2f} tokens/second")
        print(f"Raw Input Prompt Length: {len(llm_tokenizer(prompt)['input_ids'])} tokens")
        print("-"*92)
    print("="*80 + "\n")

# Run Example 1: HR Policy query
ask_question("What is the home office stipend limit and how do I submit a claim?", debug=True)

# Run Example 2: IT troubleshooting query
ask_question("How do I troubleshoot VPN connection errors and flush dns?", debug=True)

# Run Example 3: Compliance query
ask_question("What is the limit for gifts registry and reporting bribery?", debug=True)


## 13. Final Evaluation & Conclusions

### Key System Findings

1. **Preprocessing Impact:**
   - Text segmentation directly sets the search resolution boundary.
   - For corporate wikis, character chunk size of **200** with an overlap of **50** balances precision and context availability.

2. **Retrieval Latency:**
   - FAISS `IndexFlatIP` provides exact search but scale linearly.
   - FAISS `IndexIVFFlat` cuts query processing costs through space clustering, but requires data distribution training.

3. **LLM Quantization Performance:**
   - **FP16** loading requires **2.2 GB** for TinyLlama (and >14 GB for 7B models), which can trigger CUDA OOM errors.
   - **INT4** model loading saves **~75% VRAM**, enabling enterprise-grade pipelines to execute on cheaper consumer hardware, with minimal loss in reading comprehension.

4. **Prompt Optimization:**
   - Standard context passing can result in high **hallucination rates** for queries outside document bounds.
   - Incorporating strict system instructions, boundary rules, and fallback sentences (e.g. *"I could not find relevant information."*) successfully mitigates hallucination risk, maintaining enterprise compliance.

### Summary Production Checklist
- [ ] Migrate from CPU/GPU FAISS to a managed cloud database (e.g., **pgvector**, **Pinecone**, or **Qdrant**) for horizontal scalability.
- [ ] Integrate a **Cross-Encoder Reranker** to resolve "Lost in the Middle" errors when retrieval context is long.
- [ ] Implement a **query translation** pre-processor or switch to multilingual embeddings (e.g. `multilingual-e5-base`) for international document support.
- [ ] Setup persistent **evaluation pipelines** (using frameworks like `Ragas` or `TruLens`) to continuously track Groundedness and Context Recall.
